# Capstone — TPC-H Supply Chain Intelligence

**Dataset:** `samples.tpch` — all 8 tables

**Difficulty:** Hard

**Topics:** multi-table joins, window functions, collect_set, complex aggregations, revenue calculations, ranking

> These problems are **linked** — later problems build directly on the joins and patterns established earlier.
> Work through them in order.
>
> Business context: you are a data engineer at a global trading company. The exec team needs
> a series of progressively deeper reports on customer geography, order revenue, supplier
> performance, and late deliveries.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

orders   = spark.table("samples.tpch.orders")
lineitem = spark.table("samples.tpch.lineitem")
customer = spark.table("samples.tpch.customer")
nation   = spark.table("samples.tpch.nation")
region   = spark.table("samples.tpch.region")
part     = spark.table("samples.tpch.part")
supplier = spark.table("samples.tpch.supplier")
partsupp = spark.table("samples.tpch.partsupp")

---
## Problem 1 — Customer Geography

Join `customer` → `nation` → `region` to build a full geographic profile of the customer base.

For each combination of `region` and `market_segment`, compute:
- `customer_count` — number of customers
- `avg_account_balance` — average `c_acctbal`, rounded to 2 decimal places

Sort by `region` ascending, then `customer_count` descending.

**Expected output columns:** `region`, `market_segment`, `customer_count`, `avg_account_balance`

In [ ]:
# Problem 1 - write your solution here
# Assign result to: result_1

result_1 = None  # replace this

In [ ]:
# Tests for Problem 1
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'region' in cols, "Missing column: region"
assert 'market_segment' in cols, "Missing column: market_segment"
assert 'customer_count' in cols, "Missing column: customer_count"
assert 'avg_account_balance' in cols, "Missing column: avg_account_balance"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, "Got 0 rows"
assert cnt <= 25, f"Expected at most 25 rows (5 regions x 5 segments), got {cnt}"
print(f"Problem 1 passed ({cnt} rows)")

---
## Problem 2 — Revenue by Nation

Building on the nation and region join from Problem 1, now bring in `orders` to compute revenue.

Join `orders` → `customer` → `nation` → `region`.

Per nation, compute:
- `total_revenue` — sum of `o_totalprice`, rounded to 2 decimal places
- `order_count` — number of orders
- `avg_order_value` — average `o_totalprice`, rounded to 2 decimal places

Include the region name as `region`.
Sort by `total_revenue` descending.

**Expected output columns:** `n_name`, `region`, `total_revenue`, `order_count`, `avg_order_value`

In [ ]:
# Problem 2 - write your solution here
# Assign result to: result_2

result_2 = None  # replace this

In [ ]:
# Tests for Problem 2
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'n_name' in cols, "Missing column: n_name"
assert 'region' in cols, "Missing column: region"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'order_count' in cols, "Missing column: order_count"
cnt = result_2.count()
assert cnt == 25, f"Expected 25 rows (one per nation), got {cnt}"
revenues = [r['total_revenue'] for r in result_2.collect()]
assert revenues == sorted(revenues, reverse=True), "Must be sorted by total_revenue descending"
print(f"Problem 2 passed ({cnt} nations)")

---
## Problem 3 — Late Delivery Analysis

A line item is **late** when `l_receiptdate > l_commitdate`.

Join `lineitem` → `orders`. For each combination of `o_orderpriority` and year (extracted from `l_shipdate`), compute:
- `late_item_count` — number of late line items
- `total_item_count` — total line items in that group
- `late_rate_pct` — `late_item_count / total_item_count * 100`, rounded to 2 decimal places

Only include years 1993–1997. Sort by `o_orderpriority` ascending, then `year` ascending.

**Expected output columns:** `o_orderpriority`, `year`, `late_item_count`, `total_item_count`, `late_rate_pct`

In [ ]:
# Problem 3 - write your solution here
# Assign result to: result_3

result_3 = None  # replace this

In [ ]:
# Tests for Problem 3
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'o_orderpriority' in cols, "Missing column: o_orderpriority"
assert 'year' in cols, "Missing column: year"
assert 'late_rate_pct' in cols, "Missing column: late_rate_pct"
cnt = result_3.count()
assert cnt > 0, "Got 0 rows"
max_pct = result_3.agg(F.max('late_rate_pct')).collect()[0][0]
assert max_pct <= 100, f"late_rate_pct should be <= 100, got {max_pct}"
assert max_pct > 0, "Expected some late deliveries"
print(f"Problem 3 passed ({cnt} rows)")

---
## Problem 4 — Supplier Ranking within Nation

Using the same nation join pattern from Problem 2, now analyse suppliers.

Join `lineitem` → `supplier` → `nation`.

Revenue per supplier = `sum(l_extendedprice * (1 - l_discount))`.

Use a **window function** to rank each supplier within their nation by total revenue (rank 1 = highest).

Return only the **top 3 suppliers per nation** (rank <= 3).

Sort by `n_name` ascending, then `nation_rank` ascending.

**Expected output columns:** `s_name`, `n_name`, `total_revenue`, `nation_rank`

In [ ]:
# Problem 4 - write your solution here
# Assign result to: result_4

result_4 = None  # replace this

In [ ]:
# Tests for Problem 4
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 's_name' in cols, "Missing column: s_name"
assert 'n_name' in cols, "Missing column: n_name"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'nation_rank' in cols, "Missing column: nation_rank"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
max_rank = result_4.agg(F.max('nation_rank')).collect()[0][0]
assert max_rank <= 3, f"nation_rank should be <= 3, got {max_rank}"
print(f"Problem 4 passed ({cnt} rows, max rank = {max_rank})")

---
## Problem 5 — Part Type Revenue by Market Segment

Join `lineitem` → `part` → `orders` → `customer`.

For each `c_mktsegment`, compute revenue per `p_type` (use the full type string from `part`).
Revenue = `sum(l_extendedprice * (1 - l_discount))`.

Use a window function to **rank part types within each market segment** by revenue.
Keep only the **top 3 part types per segment** (rank <= 3).

Use `F.collect_list()` to produce a single row per market segment with an array of the top 3 part type names.

**Expected output columns:** `c_mktsegment`, `top_part_types` (array), `segment_total_revenue`

In [ ]:
# Problem 5 - write your solution here
# Assign result to: result_5

result_5 = None  # replace this

In [ ]:
# Tests for Problem 5
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'c_mktsegment' in cols, "Missing column: c_mktsegment"
assert 'top_part_types' in cols, "Missing column: top_part_types"
assert 'segment_total_revenue' in cols, "Missing column: segment_total_revenue"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert 1 <= cnt <= 5, f"Expected one row per market segment (1-5), got {cnt}"
# Each row should have an array with 1-3 part types
from pyspark.sql.types import ArrayType
schema_dict = {f.name: f.dataType for f in result_5.schema.fields}
top_col = [c for c in result_5.columns if 'top' in c.lower()][0]
assert isinstance(schema_dict[top_col], ArrayType), f"{top_col} must be an ArrayType"
print(f"Problem 5 passed ({cnt} market segments)")

---
## Problem 6 — Discount Impact by Shipping Mode

The finance team wants to understand how much revenue is lost to discounts, broken down by shipping mode.

From `lineitem`, compute for each `l_shipmode`:
- `revenue_with_discount` = `sum(l_extendedprice * (1 - l_discount))`, rounded to 2 dp
- `revenue_without_discount` = `sum(l_extendedprice)`, rounded to 2 dp
- `discount_impact` = `revenue_without_discount - revenue_with_discount`, rounded to 2 dp
- `discount_impact_pct` = `discount_impact / revenue_without_discount * 100`, rounded to 2 dp

Sort by `discount_impact` descending.

**Expected output columns:** `l_shipmode`, `revenue_with_discount`, `revenue_without_discount`, `discount_impact`, `discount_impact_pct`

In [ ]:
# Problem 6 - write your solution here
# Assign result to: result_6

result_6 = None  # replace this

In [ ]:
# Tests for Problem 6
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'l_shipmode' in cols, "Missing column: l_shipmode"
assert 'discount_impact' in cols, "Missing column: discount_impact"
assert 'discount_impact_pct' in cols, "Missing column: discount_impact_pct"
cnt = result_6.count()
assert cnt > 0, "Got 0 rows"
# revenue_without_discount should always be >= revenue_with_discount
rows = result_6.collect()
for r in rows:
    assert r['revenue_without_discount'] >= r['revenue_with_discount'],         f"revenue_without_discount ({r['revenue_without_discount']}) < revenue_with_discount for {r['l_shipmode']}"
print(f"Problem 6 passed ({cnt} shipping modes)")

---
## Problem 7 — Full Order Intelligence Report

The exec team needs a single comprehensive report on **high-value orders** (where `o_totalprice > 200000`).

Join: `orders` → `customer` → `nation` (customer side) + `lineitem` → `supplier` → `nation` (supplier side) + `part`

Per order, compute:
- `customer_name` — `c_name`
- `customer_nation` — the customer's nation name
- `line_item_count` — number of line items in the order
- `order_revenue` — `sum(l_extendedprice * (1 - l_discount))`, rounded to 2 dp
- `supplier_nations` — `collect_set` of distinct supplier nation names
- `part_brands` — `collect_set` of distinct `p_brand` values in the order

Filter to `o_totalprice > 200000`. Sort by `order_revenue` descending. Limit to 50 rows.

**Expected output columns:** `o_orderkey`, `customer_name`, `customer_nation`, `line_item_count`, `order_revenue`, `supplier_nations`, `part_brands`

In [ ]:
# Problem 7 - write your solution here
# Assign result to: result_7

result_7 = None  # replace this

In [ ]:
# Tests for Problem 7
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'o_orderkey' in cols, "Missing column: o_orderkey"
assert 'customer_name' in cols, "Missing column: customer_name"
assert 'supplier_nations' in cols, "Missing column: supplier_nations"
assert 'part_brands' in cols, "Missing column: part_brands"
cnt = result_7.count()
assert 0 < cnt <= 50, f"Expected 1-50 rows, got {cnt}"
from pyspark.sql.types import ArrayType
schema_dict = {f.name: f.dataType for f in result_7.schema.fields}
assert isinstance(schema_dict['supplier_nations'], ArrayType), "supplier_nations must be ArrayType"
assert isinstance(schema_dict['part_brands'], ArrayType), "part_brands must be ArrayType"
print(f"Problem 7 passed ({cnt} high-value orders)")